# Session 10 — Evaluating language models

Every session so far has measured something. This one is about the measurements themselves: what a benchmark score depends on besides the model, how precise it is, whether the model might have seen the test, how far a language model can be trusted as a judge, and how to check that a change did not break anything. We use MMLU, a standard multiple-choice benchmark, and three Qwen2.5 models: 0.5B base, 0.5B instruct and 1.5B instruct.

**By the end of the session you should be able to:**

1. score a model on a multiple-choice benchmark, give the score with an interval, and show how much the prompt format moves it;
2. run a simple contamination probe and say what it can and cannot show;
3. measure a language-model judge's position and length bias by swapping the order of the answers, and use per-item comparisons to catch regressions an average hides.

**To run it yourself** you need the packages from Session 6. The notebook downloads the MMLU test and dev splits (3.5 MB, MIT licence) and uses the three Qwen2.5 models from Session 6. Scoring 9,000 prompts took about 10 minutes on our laptop GPU and is cached in `cache/s10_lp.pt`; the whole notebook took 631 s with that cache. It is not practical on a CPU.

In [1]:
import os, re, json, math, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from aij_common import DEVICE, set_seed
import torch, torch.nn.functional as F
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
from huggingface_hub.utils import logging as hub_logging
logging.set_verbosity_error(); hub_logging.set_verbosity_error(); set_seed(0)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", padding_side="left")       # the three Qwen2.5 models share this tokenizer
MODELS = {"0.5B base": "Qwen/Qwen2.5-0.5B", "0.5B instruct": "Qwen/Qwen2.5-0.5B-Instruct", "1.5B instruct": "Qwen/Qwen2.5-1.5B-Instruct"}
def load(name): return AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16).to(DEVICE).eval()
mmlu = pd.read_parquet(hf_hub_download("cais/mmlu", "all/test-00000-of-00001.parquet", repo_type="dataset")); dev = pd.read_parquet(hf_hub_download("cais/mmlu", "all/dev-00000-of-00001.parquet", repo_type="dataset"))
test = mmlu.sample(1000, random_state=0).reset_index(drop=True)
print(f"MMLU test: {len(mmlu):,} questions in {mmlu.subject.nunique()} subjects; we use a random 1,000 ({test.subject.nunique()} subjects). Five worked examples per subject come from the dev split ({len(dev)} questions).")
print(test.iloc[0].question, "\n", list(test.iloc[0].choices), "answer:", "ABCD"[test.iloc[0].answer])

MMLU test: 14,042 questions in 57 subjects; we use a random 1,000 (57 subjects). Five worked examples per subject come from the dev split (285 questions).
Congress enacts a $100 tax on the sale of any handgun to a private individual not for use in law enforcement or military duties. Will this new handgun tax survive a constitutional challenge? 
 ['Yes, if Congress could have banned possession of handguns outright.', 'Yes, if the dominant intent of Congress was that the tax would produce revenue.', 'No, if the tax does not result in a significant collection of revenue.', 'No, because the tax is clearly intended as a penalty on handgun ownership.'] answer: B


## 1. Scoring a benchmark

MMLU has 14,042 four-option questions in 57 subjects, from school mathematics to law. We use a random 1,000. The model never writes an answer: we read the probability it gives to each of the letters A to D as its next token and take the largest. We do this with three prompt formats:

- **zero-shot**: a header naming the subject, the question, the options and "Answer:";
- **five-shot**: the same, after five solved questions from the same subject;
- **chat**: the question in the chat template, asking for the letter only.

Scoring 3 models × 3 formats × 1,000 questions is the slow step, so the scores are cached.

In [2]:
LETTERS = "ABCD"
def block(r, with_answer):
    s = r.question.strip() + "\n" + "\n".join(f"{l}. {c}" for l, c in zip(LETTERS, r.choices)) + "\nAnswer:"
    return s + f" {LETTERS[r.answer]}\n\n" if with_answer else s
def prompt(r, fmt):
    head = f"The following are multiple choice questions (with answers) about {r.subject.replace('_', ' ')}.\n\n"
    if fmt == "zero-shot": return head + block(r, False)
    if fmt == "five-shot": return head + "".join(block(d, True) for d in dev[dev.subject == r.subject].head(5).itertuples()) + block(r, False)
    return tok.apply_chat_template([{"role": "user", "content": block(r, False)[:-len("Answer:")] + "Reply with the letter of the correct answer only."}], tokenize=False, add_generation_prompt=True)
@torch.no_grad()
def letter_logprobs(m, texts, fmt, batch=8):
    """Log-probabilities the model gives to the four answer letters as its next token."""
    ids = [tok.convert_tokens_to_ids(("Ġ" if fmt != "chat" else "") + l) for l in LETTERS]      # " A" after "Answer:", plain "A" at the start of a chat reply
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], return_tensors="pt", padding=True).to(DEVICE)
        out.append(m(**enc, logits_to_keep=1).logits[:, -1].float().log_softmax(-1)[:, ids].cpu())      # only the last position's logits are computed
    return torch.cat(out)
FORMATS = ["zero-shot", "five-shot", "chat"]
lp, t0 = {}, time.time()
if os.path.exists("cache/s10_lp.pt"): lp = torch.load("cache/s10_lp.pt"); print("scores loaded from cache/s10_lp.pt")
else:
    for name, path in MODELS.items():
        m = load(path)
        for fmt in FORMATS: lp[(name, fmt)] = letter_logprobs(m, [prompt(r, fmt) for r in test.itertuples()], fmt)
        del m; torch.cuda.empty_cache()
    torch.save(lp, "cache/s10_lp.pt"); print(f"3 models x 3 prompt formats x 1,000 questions scored in {time.time() - t0:.0f} s")
truth = torch.tensor(test.answer.values)
print(f"five-shot prompts are {np.mean([len(tok(prompt(r, 'five-shot')).input_ids) for r in test.head(100).itertuples()]):.0f} tokens on average")

scores loaded from cache/s10_lp.pt


five-shot prompts are 622 tokens on average


Next, accuracy with a 95% interval from bootstrap resampling of the 1,000 questions, the share of probability that falls on the four letters at all, and how often the most popular letter is chosen.

> **Pause:** the base model has never been trained on chat conversations. What will happen to its score in the chat format?

In [3]:
def ci(correct, reps=2000, seed=0):
    g = np.random.default_rng(seed); n = len(correct); b = [correct[g.integers(0, n, n)].mean() for _ in range(reps)]
    return np.percentile(b, [2.5, 97.5])
rows = []
for name in MODELS:
    for fmt in FORMATS:
        c = (lp[(name, fmt)].argmax(1) == truth).numpy(); lo, hi = ci(c)
        p_letters = lp[(name, fmt)].exp().sum(1)
        rows.append({"model": name, "prompt": fmt, "accuracy": c.mean(), "95% interval": f"{lo:.3f}-{hi:.3f}", "prob. on the 4 letters": p_letters.mean().item(), "most chosen letter share": np.bincount(lp[(name, fmt)].argmax(1).numpy(), minlength=4).max() / len(c)})
acc = pd.DataFrame(rows); display(acc.round(3))
print(f"correct answers are spread {np.bincount(truth.numpy()) / len(truth)} over A-D")

,model,prompt,accuracy,95% interval,prob. on the 4 letters,most chosen letter share
0,0.5B base,zero-shot,0.460,0.430-0.490,0.997,0.323
1,0.5B base,five-shot,0.453,0.422-0.484,0.998,0.271
2,0.5B base,chat,0.229,0.204-0.255,0.120,0.999
3,0.5B instruct,zero-shot,0.442,0.412-0.472,0.999,0.382
4,0.5B instruct,five-shot,0.464,0.434-0.496,1.000,0.284
5,0.5B instruct,chat,0.411,0.379-0.443,0.930,0.444
6,1.5B instruct,zero-shot,0.586,0.556-0.617,0.999,0.268
7,1.5B instruct,five-shot,0.594,0.563-0.625,1.000,0.295
8,1.5B instruct,chat,0.568,0.538-0.600,0.998,0.287


correct answers are spread [0.229 0.252 0.243 0.276] over A-D


The 1.5B instruct model scores 0.586 to 0.594 in all three formats, and the two 0.5B models 0.41 to 0.46 in the formats they handle. Each interval is about ±3 points: 1,000 questions cannot separate two models closer than that.

The format matters more than the gap between the two 0.5B models. In the chat format the base model scores 0.229, below the 0.25 of a random guess. Only 12% of its probability goes to the four letters, and it picks the same letter for 99.9% of questions. The model has not got worse at anything; the format asks for something it was not trained to do. The instruct model is less affected (0.411) but still loses 5 points against five-shot. Instruction tuning did not add knowledge: the base model scores 0.460 zero-shot, the instruct model 0.442.

Published scores for these models use other prompts, other answer extraction and all 14,042 questions, so they differ from ours. Comparisons are only meaningful within one setup.

In [4]:
a = (lp[("1.5B instruct", "five-shot")].argmax(1) == truth).numpy().astype(int); b = (lp[("0.5B instruct", "five-shot")].argmax(1) == truth).numpy().astype(int)
g = np.random.default_rng(0); diffs = np.array([(a[i] - b[i]).mean() for i in (g.integers(0, 1000, 1000) for _ in range(2000))])
print(f"1.5B vs 0.5B instruct, five-shot, same 1,000 questions: difference {a.mean() - b.mean():+.3f}, 95% interval {np.percentile(diffs, 2.5):+.3f} to {np.percentile(diffs, 97.5):+.3f}")
print(f"they disagree on {(a != b).sum()} questions: 1.5B right and 0.5B wrong on {((a == 1) & (b == 0)).sum()}, the reverse on {((a == 0) & (b == 1)).sum()}")
by_subj = test.assign(right=a).groupby("subject").right.agg(["mean", "size"]).sort_values("mean")
print(f"by subject (1.5B, five-shot): lowest {by_subj.index[0]} {by_subj['mean'].iloc[0]:.2f} (n={by_subj['size'].iloc[0]}), highest {by_subj.index[-1]} {by_subj['mean'].iloc[-1]:.2f} (n={by_subj['size'].iloc[-1]}); median subject size {by_subj['size'].median():.0f} questions")

1.5B vs 0.5B instruct, five-shot, same 1,000 questions: difference +0.130, 95% interval +0.099 to +0.165
they disagree on 308 questions: 1.5B right and 0.5B wrong on 219, the reverse on 89
by subject (1.5B, five-shot): lowest high_school_computer_science 0.00 (n=4), highest medical_genetics 1.00 (n=2); median subject size 13 questions


The comparison between the two instruct models is paired: both answered the same 1,000 questions, so the bootstrap resamples questions and keeps the pairs together. The 1.5B model is ahead by 13.0 points (interval 9.9 to 16.5). They disagree on 308 questions; the 1.5B model wins 219 of them and the 0.5B model 89. A larger model is better on average and still worse on many individual questions.

Per subject, a random 1,000 questions leave a median of 13 per subject, so per-subject scores mean little: the "lowest" subject has 4 questions and the "highest" 2.

## 2. Has the model seen the test?

MMLU has been on the web since 2020, and web text is what these models are trained on. If a model has read the test questions, its score measures memory as much as ability. Training data is rarely published, so contamination has to be probed from outside.

One probe uses the order of the options. If the base model has read a question with its options in their published order, that order should look more familiar, so it should have lower loss than the same options shuffled. We compare the published order with 5 random other orders on 500 questions, leaving out questions with options such as "all of the above" whose position matters.

In [5]:
@torch.no_grad()
def text_loss(m, texts, batch=4):
    """Mean next-token loss of each text (at most its first 512 tokens), in nats per token."""
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
        logits = m(**enc).logits[:, :-1].float(); tgt = enc.input_ids[:, 1:].masked_fill(enc.attention_mask[:, 1:] == 0, -100)
        out.append(F.cross_entropy(logits.transpose(1, 2), tgt, ignore_index=-100, reduction="none").sum(1).cpu() / (tgt != -100).sum(1).cpu())
    return torch.cat(out)
def with_order(r, order):
    return r.question.strip() + "\n" + "\n".join(f"{l}. {r.choices[j]}" for l, j in zip(LETTERS, order))
probe = test[test.choices.apply(lambda c: not any(re.search(r"(all|none|both) of the (above|these)|^[A-D] and [A-D]$", x, re.I) for x in c))].head(500)
g = np.random.default_rng(1); others = [list(q) for q in __import__("itertools").permutations(range(4)) if list(q) != [0, 1, 2, 3]]      # the 23 other orders
perms = [[others[j] for j in g.choice(23, 5, replace=False)] for _ in range(len(probe))]
base = load(MODELS["0.5B base"])
orig = text_loss(base, [with_order(r, [0, 1, 2, 3]) for r in probe.itertuples()])
shuf = torch.stack([text_loss(base, [with_order(r, p[k]) for r, p in zip(probe.itertuples(), perms)]) for k in range(5)], 1)
wins = (orig < shuf.min(1).values).float().mean().item()
print(f"{len(probe)} questions without 'all of the above' style options; loss with the original option order {orig.mean():.3f} nats/token, shuffled orders {shuf.mean():.3f}")
print(f"the original order has the lowest loss of 6 orders in {wins:.0%} of questions; with no memory of the order we expect about 1 in 6 = 17% (more if the original order is simply more natural, e.g. numbers in ascending order)")
course = [json.loads(l)["text"] for l in open("data/basic_course_chunks.jsonl")]
course = [c[:600] for c in course if len(c) > 300][:200]; mm = [with_order(r, [0, 1, 2, 3])[:600] for r in probe.head(200).itertuples()]
print(f"for comparison, loss on text the model cannot have seen (the basic course, written in 2026): {text_loss(base, course).mean():.3f} nats/token; on MMLU questions: {text_loss(base, mm).mean():.3f}")
del base; torch.cuda.empty_cache()

500 questions without 'all of the above' style options; loss with the original option order 2.376 nats/token, shuffled orders 2.393
the original order has the lowest loss of 6 orders in 38% of questions; with no memory of the order we expect about 1 in 6 = 17% (more if the original order is simply more natural, e.g. numbers in ascending order)


for comparison, loss on text the model cannot have seen (the basic course, written in 2026): 3.170 nats/token; on MMLU questions: 2.428


The published order has the lowest loss of the six in 38% of questions, more than twice the 17% expected if the model had no preference among orders. As a reference, the same model's loss on MMLU questions is 2.428 nats per token, against 3.170 on the basic-course text written this year, which it cannot have seen.

This is evidence, not proof. Some published orders are simply more natural than any shuffle: numbers in ascending order, "true, true / true, false" patterns, or options that read as a sequence. A clean test needs questions known to be absent from training, for example new questions written after the model's training cut-off, scored the same way.

## 3. A language model as judge

For open-ended answers there is no answer key, and a common shortcut is to ask a language model which of two answers is better. We give each instruct model 400 UltraFeedback pairs from Session 6 and read the probability it gives to "A" against "B", then repeat with the two answers swapped. A judge that reads the answers gives the same verdict in both orders.

> **Pause:** how often will the 0.5B judge pick whichever answer is shown first?

In [6]:
pairs = json.load(open("cache/s06_pairs.json"))[-1000:]                             # the same validation pairs as Session 6, never used for training here
pairs = [p for p in pairs if len(p[1]) + len(p[2]) < 2500][:400]
def judge_prompt(q, a, b):
    return tok.apply_chat_template([{"role": "user", "content": f"Here is a question and two answers.\n\nQuestion: {q}\n\nAnswer A:\n{a}\n\nAnswer B:\n{b}\n\nWhich answer is better? Reply with A or B only."}], tokenize=False, add_generation_prompt=True)
@torch.no_grad()
def judge(m, items, batch=8):
    """P(the judge says A) for each (question, first, second)."""
    iA, iB = tok.convert_tokens_to_ids("A"), tok.convert_tokens_to_ids("B"); out = []
    for i in range(0, len(items), batch):
        enc = tok([judge_prompt(*x) for x in items[i:i + batch]], return_tensors="pt", padding=True).to(DEVICE)
        l = m(**enc, logits_to_keep=1).logits[:, -1].float(); out.append((l[:, iA] - l[:, iB]).sigmoid().cpu())
    return torch.cat(out)
longer_chosen = np.array([len(tok(c).input_ids) > len(tok(r).input_ids) for _, c, r in pairs])
rows = []
for name in ("0.5B instruct", "1.5B instruct"):
    m = load(MODELS[name]); pa = judge(m, [(q, c, r) for q, c, r in pairs]).numpy(); pb = judge(m, [(q, r, c) for q, c, r in pairs]).numpy(); del m; torch.cuda.empty_cache()
    first, second = pa > 0.5, pb > 0.5                                                 # chose the preferred answer when it was shown first / chose the other answer when the preferred one was second
    picks_chosen_2 = ~second; consistent = first == picks_chosen_2
    picks_longer = np.where(longer_chosen, first, ~first)
    rows.append({"judge": name, "agrees with label (preferred shown first)": first.mean(), "agrees (preferred shown second)": picks_chosen_2.mean(), "same verdict both orders": consistent.mean(),
                 "picks whichever is shown first": (np.concatenate([first, second])).mean(), "agrees, when consistent": first[consistent].mean(),
                 "picks the longer answer": picks_longer.mean()})
display(pd.DataFrame(rows).round(3))
print(f"{len(pairs)} UltraFeedback pairs (labels come from GPT-4 ratings, not people); the preferred answer is the longer one in {longer_chosen.mean():.0%} of them")

,judge,agrees with label (preferred shown first),agrees (preferred shown second),same verdict both orders,picks whichever is shown first,"agrees, when consistent",picks the longer answer
0,0.5B instruct,0.988,0.03,0.032,0.979,0.769,0.525
1,1.5B instruct,0.555,0.60,0.480,0.478,0.661,0.632


400 UltraFeedback pairs (labels come from GPT-4 ratings, not people); the preferred answer is the longer one in 53% of them


The 0.5B judge picks the first answer 97.9% of the time. It agrees with the label 0.988 of the time when the preferred answer is shown first and 0.03 when it is shown second, and gives the same verdict in both orders for 3.2% of pairs. It is not judging anything.

The 1.5B judge has almost no overall position preference (0.478 for the first answer), but it gives the same verdict in both orders for only 48% of pairs. When it is consistent it agrees with the label 66% of the time, about as often as Session 6's reward models. It prefers the longer answer 63% of the time, where the labels prefer it 53% of the time.

A judge's agreement measured in one order says little: the 0.5B judge would look excellent with the preferred answer always shown first. Always score both orders, report how often the verdicts agree, and check for a length preference.

## 4. Did the change break anything?

Models change all the time: new weights, new prompts, quantisation to save memory. Before shipping a change, compare old and new item by item. We quantise the 0.5B instruct model's weights to 4 bits (one scale per 128 weights, as in Session 9) and compare it with the original on the same 1,000 MMLU questions and on 12 small format checks with exact expected answers (a number only, a given JSON object, one word).

In [7]:
def quantise_int4(m, group=128):
    """Round every weight matrix inside the blocks to 4 bits with one scale per 128 weights (as in Session 9)."""
    with torch.no_grad():
        for mod in m.model.layers.modules():
            if isinstance(mod, torch.nn.Linear):
                w = mod.weight.float().view(mod.weight.shape[0], -1, group); s = w.abs().amax(-1, keepdim=True).clamp(min=1e-8) / 7
                mod.weight.copy_((torch.round(w / s).clamp(-8, 7) * s).view_as(mod.weight).to(mod.weight.dtype))
    return m
CHECKS = [("What is 17 + 25? Reply with the number only.", lambda o: o.strip().rstrip(".") == "42"), ("What is 9 times 8? Reply with the number only.", lambda o: o.strip().rstrip(".") == "72"),
          ("Reply with a JSON object with the keys name and age for Alice, who is 30. JSON only.", lambda o: _json(o) == {"name": "Alice", "age": 30}),
          ("Reply with a JSON object with the key colours holding a list of the three primary colours of light, in lower case. JSON only.", lambda o: sorted((_json(o) or {}).get("colours", [])) == ["blue", "green", "red"]),
          ("Translate 'thank you' into French. Reply with the translation only.", lambda o: o.strip().lower().rstrip(".!") == "merci"),
          ("What is the capital of Australia? One word.", lambda o: o.strip().rstrip(".") == "Canberra"), ("Write the word 'robot' backwards. Reply with that only.", lambda o: o.strip().lower().rstrip(".") == "tobor"),
          ("List the first five prime numbers, separated by commas, nothing else.", lambda o: re.sub(r"\s", "", o).rstrip(".") == "2,3,5,7,11"),
          ("Is 91 a prime number? Answer yes or no.", lambda o: o.strip().lower().startswith("no")), ("How many legs does a spider have? Reply with the number only.", lambda o: o.strip().rstrip(".") == "8"),
          ("Reply with the word 'ready' in capital letters and nothing else.", lambda o: o.strip().rstrip(".!") == "READY"), ("What is 2 to the power of 10? Reply with the number only.", lambda o: o.strip().replace(",", "").rstrip(".") == "1024")]
def _json(o):
    try: return json.loads(re.search(r"\{.*\}", o, re.S).group(0))
    except Exception: return None
@torch.no_grad()
def run_checks(m):
    enc = tok([tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q, _ in CHECKS], return_tensors="pt", padding=True).to(DEVICE)
    outs = tok.batch_decode(m.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id)[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return np.array([bool(f(o)) for o, (_, f) in zip(outs, CHECKS)]), outs
m = load(MODELS["0.5B instruct"]); ok_before, out_before = run_checks(m)
m = quantise_int4(m); lp_q = letter_logprobs(m, [prompt(r, "five-shot") for r in test.itertuples()], "five-shot"); ok_after, out_after = run_checks(m); del m; torch.cuda.empty_cache()
before = (lp[("0.5B instruct", "five-shot")].argmax(1) == truth).numpy(); after = (lp_q.argmax(1) == truth).numpy()
print(f"MMLU, 1,000 questions, five-shot: bf16 {before.mean():.3f}, int4 {after.mean():.3f}; answers that changed: {(lp_q.argmax(1) != lp[('0.5B instruct', 'five-shot')].argmax(1)).sum().item()}, "
      f"right -> wrong {(before & ~after).sum()}, wrong -> right {(~before & after).sum()}")
print(f"{len(CHECKS)} format checks: {ok_before.sum()} pass in bf16, {ok_after.sum()} after int4")
for (q, _), a, b, x, y in zip(CHECKS, ok_before, ok_after, out_before, out_after):
    if a != b: print(f"  changed: {q}\n    bf16: {x!r}\n    int4: {y!r}")

MMLU, 1,000 questions, five-shot: bf16 0.464, int4 0.354; answers that changed: 475, right -> wrong 204, wrong -> right 94
12 format checks: 5 pass in bf16, 4 after int4
  changed: Reply with a JSON object with the keys name and age for Alice, who is 30. JSON only.
    bf16: '```json\n{\n  "name": "Alice",\n  "age": 30\n}\n```'
    int4: '```json\n{"name": "Alice", "age": "30"}\n```'
  changed: List the first five prime numbers, separated by commas, nothing else.
    bf16: '2, 3, 5, 7, 11'
    int4: '11, 23, 47, 61, 89'
  changed: Is 91 a prime number? Answer yes or no.
    bf16: 'Yes.'
    int4: 'No, 91 is not a prime number. It has factors other than 1 and itself (i.e., 7 and 13).'
  changed: How many legs does a spider have? Reply with the number only.
    bf16: 'A spider has eight legs.'
    int4: '8'
  changed: What is 2 to the power of 10? Reply with the number only.
    bf16: '1024'
    int4: '4096'


On MMLU the int4 model drops from 0.464 to 0.354, and the per-item view shows more: 475 of the 1,000 answers changed, 204 from right to wrong and 94 from wrong to right.

The format checks show the same thing on a small scale. Five pass before and four after, a loss of one; but five checks changed, in both directions. After quantisation the model gets the primes and $2^{10}$ wrong and writes the age in the JSON as a string, while it now answers "8" where it used to write a sentence and correctly says 91 is not prime. A single total of 5 against 4 would hide all of that. Keep a fixed set of checks, compare item by item, and read the changes.

## What to take away

- A benchmark score depends on the prompt format and the answer extraction as much as on the model. Report the setup, and compare models only within one setup.
- With 1,000 questions, an interval is about ±3 points. Compare models on the same items, with a paired interval, and look at the items where they disagree.
- Contamination is hard to rule out. A probe such as option order gives evidence; new questions written after the training cut-off give a clean test.
- A language-model judge must be checked in both answer orders. A small judge can be almost pure position bias, and a better one still prefers longer answers.
- Check changes item by item against a fixed set of tests. An average that moves a little can hide many answers that changed.

---

# Going deeper

Reading for after the session.

## MMLU

Hendrycks et al. (2021), *Measuring Massive Multitask Language Understanding*, introduced the benchmark. Published scores differ in prompt wording, in the number of solved examples, and in whether the answer is read from the letter probabilities, as here, or from generated text. Several cleaned and harder versions now exist, because some original questions have wrong or ambiguous answers.

## Contamination

Probes such as option order, completing a question from its first half, or comparing the loss on original and paraphrased questions can show that a model has seen a test, but none can show that it has not. The most reliable defences are benchmarks with questions written after a model's training cut-off, and held-out test sets that are never published.

## Judges

Zheng et al. (2023), *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*, measured position bias, a preference for longer answers and a preference for the judge's own style, and proposed swapping positions and counting inconsistent verdicts as ties. Even strong judges should be checked against a sample of human judgements on the task at hand.

## Regression tests

The format checks here are a small version of what teams run before every release: a fixed suite of prompts with checkable answers, scored item by item, with every change read by a person. Keep the suite under version control, add a test for every bug found in use, and never tune prompts or models on it directly, or it stops measuring anything.